In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
def load_data_and_chunks(filename,chunksize=40,max_chunks=1000):
    with open(filename,'r') as f:
        raw_data = f.read().upper()
    mapp = {'A':0,'C':1,'G':2,'T':3}
    sequence = [mapp[c] for c in raw_data if c in mapp]
    chunks = [sequence[i:i + chunksize] for i in range(0, len(sequence), chunksize)]
    return np.array(chunks[:max_chunks])

In [3]:
def build_kraus(parameter):
    kraus_op = []
    for i in range(4):
        matrix = np.array([[parameter[i][0],parameter[i][1]],
                          [parameter[i][2],parameter[i][3]]])
        kraus_op.append(matrix)
    return kraus_op

In [4]:
def calculating_loss_of_chunk(parameter,sequence):
    density_matrix = np.array([[1.0,0.0],[0.0,0.0]],requires_grad = False)
    kraus_op = build_kraus(parameter)
    log_prob_total = 0.0
    for c in sequence:
        k = kraus_op[c]
        new_density_matrix = k@density_matrix@k.T
        probability = np.trace(new_density_matrix)
        log_prob_total+=np.log(probability+1e-9)
        density_matrix = new_density_matrix/(probability+1e-9)
    return log_prob_total

In [5]:
def cost_function(parameter,batch):
    total_loss = 0.0
    for seq in batch:
        total_loss-=calculating_loss_of_chunk(parameter,seq)
    return total_loss/len(batch)

In [6]:
def hidden_quantum_markov_model(filename):
    dataset = load_data_and_chunks(filename)
    np.random.seed(40)
    parameter = np.random.uniform(-0.5,0.5,size=(4,4),requires_grad=True)
    optimiser = qml.AdamOptimizer(stepsize=0.2)
    np.random.shuffle(dataset)
    train_ratio = 0.8
    total_chunks = len(dataset)
    index = int(np.floor(total_chunks*train_ratio))
    test_set = dataset[index:]
    dataset = dataset[:index]
    batch_size = 15
    for i in range(100):
        np.random.shuffle(dataset)
        batch = [dataset[j:j+batch_size] for j in range(0,len(dataset),batch_size)]
        loss = 0.0
        for b in batch:
            parameter,b_loss = optimiser.step_and_cost(lambda p:cost_function(p,b),parameter)
            loss+=b_loss
        if i%20==0:
            print(f'average loss for a batch is {loss}')
    print(f'loss for test set is {cost_function(parameter,test_set)}')
    return parameter

In [7]:
parameter = hidden_quantum_markov_model('chimpanzee.txt')

average loss for a batch is -5430.053284330511
average loss for a batch is -19760.48441057706
average loss for a batch is -23562.972925644168
average loss for a batch is -26142.669275959135
average loss for a batch is -27796.002261393594
loss for test set is -538.454423305023
